In [1]:
import numpy as np  
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix



In [2]:
df = pd.read_csv('MedTech_Masterchart - final (1).csv')


In [3]:
df.shape

(226, 329)

In [4]:
df.head()

,S.no,Age,weight,height,Pulse rate,Pain,Site,staging (TNM),Vata,Pitta,...,K_Q62.2,K_Q63,K_Q64,K_Q65_final,K_Q65,K_Q65.1,K_Q66_final,K_Q66,K_Q66.1,K_Q67
0,1,60,75,168,73,5,Hypopharyngeal,T2N2cM0,25.2,24.4,...,0.5,1,1,1,0,1,0,0,0,1
1,2,61,65,165,91,6,Glottis,T3N1M0,38.2,37.9,...,0.0,0,0,1,0,1,1,0,1,0
2,3,56,70,170,88,2,Glottis,T2N0M0,29.1,32.2,...,0.5,1,1,1,0,1,0,0,0,0
3,4,46,67,185,72,7,Oropharyngeal,T4bN2cM0,29.5,24.0,...,0.5,1,1,1,0,1,0,0,0,1
4,5,63,67,185,85,3,Oropharyngeal,T2N0M0,38.9,21.7,...,0.5,1,1,1,0,1,0,0,0,1


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226 entries, 0 to 225
Columns: 329 entries, S.no to K_Q67
dtypes: float64(54), int64(273), object(2)
memory usage: 581.0+ KB


In [6]:
print(df.isnull().sum().sort_values(ascending=False).head(30))


P_Q16          141
P_Q19          105
V_Q9            61
P_Q9            61
V_Q18           55
K_Q4            11
V_Q4            11
P_Q13           10
K_Q13           10
P_Q56.3          0
P_Q59            0
P_Q59_final      0
P_Q58            0
P_Q57            0
P_Q56.4          0
P_Q56            0
P_Q56.2          0
P_Q56.1          0
P_Q59.2          0
P_Q56_final      0
P_Q55            0
P_Q54            0
P_Q53.1          0
P_Q53            0
P_Q59.1          0
P_Q62_final      0
P_Q60            0
P_Q61            0
K_Q3             0
K_Q2.1           0
dtype: int64


In [7]:
# We'll handle missing values inside model pipelines to avoid leakage.
print('Missing values (top 30):')
print(df.isnull().sum().sort_values(ascending=False).head(30))


Missing values (top 30):
P_Q16          141
P_Q19          105
V_Q9            61
P_Q9            61
V_Q18           55
K_Q4            11
V_Q4            11
P_Q13           10
K_Q13           10
P_Q56.3          0
P_Q59            0
P_Q59_final      0
P_Q58            0
P_Q57            0
P_Q56.4          0
P_Q56            0
P_Q56.2          0
P_Q56.1          0
P_Q59.2          0
P_Q56_final      0
P_Q55            0
P_Q54            0
P_Q53.1          0
P_Q53            0
P_Q59.1          0
P_Q62_final      0
P_Q60            0
P_Q61            0
K_Q3             0
K_Q2.1           0
dtype: int64


In [8]:
# Get all column names
cols = df.columns

# Columns to keep
keep_cols = []

for col in cols:
    # If it's a "_final" column → always keep
    if col.endswith('_final'):
        keep_cols.append(col)
    else:
        # Check if corresponding _final exists
        base = col.split('.')[0]  # handles K_Q65.1
        final_col = base + '_final'
        
        if final_col not in cols:
            keep_cols.append(col)

# Keep only selected columns
df = df[keep_cols]

In [9]:
df.head()

,S.no,Age,weight,height,Pulse rate,Pain,Site,staging (TNM),Vata,Pitta,...,K_Q58,K_Q59_final,K_Q60,K_Q61,K_Q62_final,K_Q63,K_Q64,K_Q65_final,K_Q66_final,K_Q67
0,1,60,75,168,73,5,Hypopharyngeal,T2N2cM0,25.2,24.4,...,0.5,1,1,0.5,1.0,1,1,1,0,1
1,2,61,65,165,91,6,Glottis,T3N1M0,38.2,37.9,...,0.5,1,0,0.0,0.0,0,0,1,1,0
2,3,56,70,170,88,2,Glottis,T2N0M0,29.1,32.2,...,0.5,1,1,0.5,1.0,1,1,1,0,0
3,4,46,67,185,72,7,Oropharyngeal,T4bN2cM0,29.5,24.0,...,0.5,1,1,0.5,1.0,1,1,1,0,1
4,5,63,67,185,85,3,Oropharyngeal,T2N0M0,38.9,21.7,...,0.5,1,1,0.5,1.0,1,1,1,0,1


In [10]:
df.shape

(226, 251)

In [11]:
print(df.columns.tolist())

['S.no', 'Age', 'weight', 'height', 'Pulse rate', 'Pain', 'Site', 'staging (TNM)', 'Vata', 'Pitta', 'Kapha', 'SF 36', 'Value_Q1', 'Value_Q2', 'Value_Q3', 'Value_Q4', 'Value_Q5', 'Value_Q6', 'Value_Q7', 'Value_Q8', 'Value_Q9', 'Value_Q10', 'Value_Q11', 'Value_Q12', 'Value_Q13', 'Value_Q14', 'Value_Q15', 'Value_Q16', 'Value_Q17', 'Value_Q18', 'Value_Q19', 'Value_Q20', 'Value_Q21', 'Value_Q22', 'Value_Q23', 'Value_Q24', 'Value_Q25', 'Value_Q26', 'Value_Q27', 'Value_Q28', 'Value_Q29', 'Value_Q30', 'Value_Q31', 'Value_Q32', 'Value_Q33', 'Value_Q34', 'Value_Q35', 'Value_Q36', 'V_Q1', 'V_Q2_final', 'V_Q3', 'V_Q4', 'V_Q5', 'V_Q6', 'V_Q7', 'V_Q8', 'V_Q9', 'V_Q10', 'V_Q11', 'V_Q12', 'V_Q13', 'V_Q14', 'V_Q15', 'V_Q16', 'V_Q17_final', 'V_Q18', 'V_Q19', 'V_Q20', 'V_Q21_final', 'V_Q22', 'V_Q23', 'V_Q24', 'V_Q25', 'V_Q26', 'V_Q27', 'V_Q28', 'V_Q29', 'V_Q30', 'V_Q31', 'V_Q32', 'V_Q33', 'V_Q34', 'V_Q35', 'V_Q36', 'V_Q37', 'V_Q38', 'V_Q39', 'V_Q40', 'V_Q41', 'V_Q42', 'V_Q43', 'V_Q44', 'V_Q45', 'V_Q46', 

In [12]:
for col in df.columns:
    print(f'{col}: {df[col].dtypes}')

S.no: int64
Age: int64
weight: int64
height: int64
Pulse rate: int64
Pain: int64
Site: object
staging (TNM): object
Vata: float64
Pitta: float64
Kapha: float64
SF 36: int64
Value_Q1: int64
Value_Q2: int64
Value_Q3: int64
Value_Q4: int64
Value_Q5: int64
Value_Q6: int64
Value_Q7: int64
Value_Q8: int64
Value_Q9: int64
Value_Q10: int64
Value_Q11: int64
Value_Q12: int64
Value_Q13: int64
Value_Q14: int64
Value_Q15: int64
Value_Q16: int64
Value_Q17: int64
Value_Q18: int64
Value_Q19: int64
Value_Q20: int64
Value_Q21: float64
Value_Q22: int64
Value_Q23: float64
Value_Q24: float64
Value_Q25: float64
Value_Q26: float64
Value_Q27: float64
Value_Q28: float64
Value_Q29: float64
Value_Q30: float64
Value_Q31: float64
Value_Q32: int64
Value_Q33: int64
Value_Q34: int64
Value_Q35: int64
Value_Q36: int64
V_Q1: int64
V_Q2_final: int64
V_Q3: int64
V_Q4: float64
V_Q5: int64
V_Q6: int64
V_Q7: int64
V_Q8: int64
V_Q9: float64
V_Q10: int64
V_Q11: int64
V_Q12: int64
V_Q13: int64
V_Q14: int64
V_Q15: int64
V_Q16: i

In [13]:
df['Site'].value_counts()

Site
Oral cavity                 62
Supraglottic                39
Glottic                     33
Hypopharyngeal              30
Oropharyngeal               23
Sinonasal carcinoma         10
Nasopharyngeal Carcinoma     8
Maxillary sinus              5
Sinonasal Carcinoma          4
Glottis                      3
Esophagus                    2
Esophageal                   1
Sinonasal chondrosarcoma     1
Oropharynx                   1
sinonasal carcinoma          1
Sinonasal Sarcoma            1
Nasopharyngeal carcinoma     1
Transglottic                 1
Name: count, dtype: int64

In [14]:
import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

def split_tnm_into_t_n(raw, keep_suffixes=False):
    """
    Extracts T and N from a raw TNM string.

    Examples:
        T2aN1bM0  ->  T2, N1   (default)
        T4bN2cM1  ->  T4, N2   (default)
        TisN0M0   ->  Tis, N0

    If keep_suffixes=True:
        T2a -> T2a
        N1b -> N1b
    """
    if pd.isna(raw):
        return np.nan, np.nan

    s = str(raw).upper().strip()
    s = re.sub(r'[\s\-_\/]+', '', s)
    s = re.sub(r'[^TNM0-9ABCIS]', '', s)

    t_match = re.search(r'T(IS|[0-4][ABC]?)', s)
    n_match = re.search(r'N([0-3][ABC]?)', s)

    if not (t_match and n_match):
        return np.nan, np.nan

    t_tok = t_match.group(1)
    n_tok = n_match.group(1)

    # T label
    if t_tok == "IS":
        t_label = "Tis"
    else:
        t_base = t_tok[0]
        t_suffix = t_tok[1:] if len(t_tok) > 1 else ""
        t_label = f"T{t_base}{t_suffix.lower()}" if keep_suffixes and t_suffix else f"T{t_base}"

    # N label
    n_base = n_tok[0]
    n_suffix = n_tok[1:] if len(n_tok) > 1 else ""
    n_label = f"N{n_base}{n_suffix.lower()}" if keep_suffixes and n_suffix else f"N{n_base}"

    return t_label, n_label


def add_tn_columns(df, stage_col="staging (TNM)", keep_suffixes=False):
    """
    Adds:
      - T_stage : encoded T label
      - N_stage : encoded N label

    Also returns the fitted encoders.
    """
    df = df.copy()

    if stage_col not in df.columns:
        raise ValueError(f"Column '{stage_col}' not found")

    parsed = df[stage_col].apply(lambda x: split_tnm_into_t_n(x, keep_suffixes=keep_suffixes))
    parsed_df = pd.DataFrame(parsed.tolist(), columns=["T_label", "N_label"], index=df.index)

    df = pd.concat([df, parsed_df], axis=1)

    # keep only rows where both T and N were successfully parsed
    mask = df["T_label"].notna() & df["N_label"].notna()
    df = df.loc[mask].copy()

    t_le = LabelEncoder()
    n_le = LabelEncoder()

    df["T_stage"] = t_le.fit_transform(df["T_label"].astype(str))
    df["N_stage"] = n_le.fit_transform(df["N_label"].astype(str))

    # remove helper string columns
    df.drop(columns=["T_label", "N_label"], inplace=True)

    return df, t_le, n_le


# Usage
df, t_le, n_le = add_tn_columns(df, stage_col="staging (TNM)", keep_suffixes=False)

print("T classes:", list(t_le.classes_))
print("N classes:", list(n_le.classes_))
print(df[["staging (TNM)", "T_stage", "N_stage"]].head())

T classes: ['T1', 'T2', 'T3', 'T4']
N classes: ['N0', 'N1', 'N2', 'N3']
  staging (TNM)  T_stage  N_stage
0       T2N2cM0        1        2
1        T3N1M0        2        1
2        T2N0M0        1        0
3      T4bN2cM0        3        2
4        T2N0M0        1        0


In [15]:
df


,S.no,Age,weight,height,Pulse rate,Pain,Site,staging (TNM),Vata,Pitta,...,K_Q60,K_Q61,K_Q62_final,K_Q63,K_Q64,K_Q65_final,K_Q66_final,K_Q67,T_stage,N_stage
0,1,60,75,168,73,5,Hypopharyngeal,T2N2cM0,25.2,24.4,...,1,0.5,1.0,1,1,1,0,1,1,2
1,2,61,65,165,91,6,Glottis,T3N1M0,38.2,37.9,...,0,0.0,0.0,0,0,1,1,0,2,1
2,3,56,70,170,88,2,Glottis,T2N0M0,29.1,32.2,...,1,0.5,1.0,1,1,1,0,0,1,0
3,4,46,67,185,72,7,Oropharyngeal,T4bN2cM0,29.5,24.0,...,1,0.5,1.0,1,1,1,0,1,3,2
4,5,63,67,185,85,3,Oropharyngeal,T2N0M0,38.9,21.7,...,1,0.5,1.0,1,1,1,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,222,61,75,176,76,3,Hypopharyngeal,T3N0M0,29.5,24.0,...,1,0.5,1.0,1,1,1,0,1,2,0
222,223,51,69,175,71,3,Oral cavity,T4aN2bM0,29.3,31.0,...,1,0.5,1.0,1,1,1,0,0,3,2
223,224,70,61,166,77,5,Glottic,T3N0M0,38.9,21.7,...,1,0.5,1.0,1,1,1,0,1,2,0
224,225,75,52,176,74,3,Oropharyngeal,T3N1M0,36.6,23.9,...,1,0.5,1.0,1,1,1,0,1,2,1


In [16]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# -----------------------------
# 1. Define numeric columns
# -----------------------------
numeric_cols = ["Pain", "weight", "height", "Pulse rate"]

# Ensure numeric conversion
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

# -----------------------------
# 2. Filter valid rows
# -----------------------------
# Now we use T_stage and N_stage instead of stage_clean
required_cols = ["T_stage", "N_stage"] + numeric_cols

mask = pd.Series(True, index=df.index)

for col in required_cols:
    mask &= df[col].notna()

df = df.loc[mask].copy()

print("After filtering:", df.shape)

# -----------------------------
# 3. Min-Max scaling (replace original columns)
# -----------------------------
scaler = MinMaxScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# -----------------------------
# Final dataset ready
# -----------------------------
print("\nFinal shape:", df.shape)
print(df[["T_stage", "N_stage"] + numeric_cols].head())

After filtering: (226, 253)

Final shape: (226, 253)
   T_stage  N_stage   Pain    weight    height  Pulse rate
0        1        2  0.500  0.681818  0.387097    0.343750
1        2        1  0.625  0.454545  0.290323    0.625000
2        1        0  0.125  0.568182  0.451613    0.578125
3        3        2  0.750  0.500000  0.935484    0.328125
4        1        0  0.250  0.500000  0.935484    0.531250


In [17]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

# ensure df is DataFrame
print(type(df))

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded = encoder.fit_transform(df[["Site"]])

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(["Site"])
)

df = pd.concat([df, encoded_df], axis=1)
df.drop("Site", axis=1, inplace=True)

<class 'pandas.core.frame.DataFrame'>


In [18]:
df

,S.no,Age,weight,height,Pulse rate,Pain,staging (TNM),Vata,Pitta,Kapha,...,Site_Oral cavity,Site_Oropharyngeal,Site_Oropharynx,Site_Sinonasal Carcinoma,Site_Sinonasal Sarcoma,Site_Sinonasal carcinoma,Site_Sinonasal chondrosarcoma,Site_Supraglottic,Site_Transglottic,Site_sinonasal carcinoma
0,1,60,0.681818,0.387097,0.343750,0.500,T2N2cM0,25.2,24.4,50.4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,61,0.454545,0.290323,0.625000,0.625,T3N1M0,38.2,37.9,23.9,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,56,0.568182,0.451613,0.578125,0.125,T2N0M0,29.1,32.2,38.8,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,46,0.500000,0.935484,0.328125,0.750,T4bN2cM0,29.5,24.0,46.5,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,63,0.500000,0.935484,0.531250,0.250,T2N0M0,38.9,21.7,39.3,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,222,61,0.681818,0.645161,0.390625,0.250,T3N0M0,29.5,24.0,46.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
222,223,51,0.545455,0.612903,0.312500,0.250,T4aN2bM0,29.3,31.0,39.7,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
223,224,70,0.363636,0.322581,0.406250,0.500,T3N0M0,38.9,21.7,39.3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
224,225,75,0.159091,0.645161,0.359375,0.250,T3N1M0,36.6,23.9,39.5,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [19]:
import pandas as pd
import numpy as np
import re

def prepare_dataset(df, mode="value"):
    """
    mode options:
    - "value" → Value_Q1 to Value_Q36
    - "vpk"   → V_Q*, P_Q*, K_Q*
    - "all"   → Value_Q* + V_Q* + P_Q* + K_Q*

    Returns:
      X        : feature matrix
      y_T      : T_stage
      y_N      : N_stage
      y_pain   : Pain
    """

    df = df.copy()

    # -----------------------------
    # 1. Clean column names
    # -----------------------------
    df.columns = df.columns.str.strip()
    df.columns = [re.sub(r'\.\d+$', '', col) for col in df.columns]
    df.columns = [col.replace('-', '_') for col in df.columns]
    df = df.loc[:, ~df.columns.duplicated()]

    # -----------------------------
    # 2. Targets
    # -----------------------------
    required_targets = ["T_stage", "N_stage", "Pain"]

    for col in required_targets:
        if col not in df.columns:
            raise ValueError(f"Target column '{col}' not found")

    y_T = pd.to_numeric(df["T_stage"], errors="coerce")
    y_N = pd.to_numeric(df["N_stage"], errors="coerce")
    y_pain = pd.to_numeric(df["Pain"], errors="coerce")

    # Keep only valid rows
    mask = y_T.notna() & y_N.notna() & y_pain.notna()
    df = df.loc[mask].copy()

    y_T = y_T.loc[mask].astype(int)
    y_N = y_N.loc[mask].astype(int)
    y_pain = y_pain.loc[mask].astype(float)

    # -----------------------------
    # 3. Select feature columns
    # -----------------------------
    if mode == "value":
        feature_cols = [col for col in df.columns if col.startswith("Value_Q")]

    elif mode == "vpk":
        feature_cols = [
            col for col in df.columns
            if col.startswith("V_Q") or col.startswith("P_Q") or col.startswith("K_Q")
        ]

    elif mode == "all":
        feature_cols = [
            col for col in df.columns
            if col.startswith("Value_Q")
            or col.startswith("V_Q")
            or col.startswith("P_Q")
            or col.startswith("K_Q")
        ]

    else:
        raise ValueError("Invalid mode. Choose from: value, vpk, all")

    # -----------------------------
    # 4. Add Site encoded columns
    # -----------------------------
    site_cols = [col for col in df.columns if col.startswith("Site_")]

    # -----------------------------
    # 5. Final feature matrix
    # -----------------------------
    final_cols = feature_cols + site_cols

    X = (
        df[final_cols]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )

    return (
        np.nan_to_num(np.asarray(X, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0),
        np.asarray(y_T),
        np.asarray(y_N),
        np.asarray(y_pain),
    )


# -----------------------------
# Example usage
# -----------------------------
X1, y_T1, y_N1, y_pain1 = prepare_dataset(df, mode="value")
X2, y_T2, y_N2, y_pain2 = prepare_dataset(df, mode="vpk")
X3, y_T3, y_N3, y_pain3 = prepare_dataset(df, mode="all")

print(X1.shape, X2.shape, X3.shape)
print("T classes:", len(np.unique(y_T1)))
print("N classes:", len(np.unique(y_N1)))
print("Pain range:", y_pain1.min(), y_pain1.max())

(226, 54) (226, 220) (226, 256)
T classes: 4
N classes: 4
Pain range: 0.0 1.0


In [20]:
y_N1

array([2, 1, 0, 2, 0, 1, 0, 1, 1, 1, 2, 3, 1, 0, 1, 2, 0, 1, 0, 3, 1, 1,
       3, 2, 0, 3, 1, 1, 1, 2, 1, 0, 0, 0, 0, 1, 2, 1, 1, 0, 2, 2, 0, 1,
       0, 1, 2, 1, 0, 2, 0, 2, 1, 1, 1, 2, 0, 0, 2, 1, 1, 1, 1, 1, 0, 0,
       0, 1, 0, 3, 1, 1, 1, 0, 0, 1, 1, 1, 2, 1, 2, 1, 2, 0, 1, 1, 2, 1,
       2, 2, 1, 1, 1, 0, 1, 0, 0, 1, 3, 3, 1, 3, 2, 0, 0, 1, 1, 1, 1, 2,
       1, 0, 2, 1, 3, 2, 1, 2, 1, 2, 1, 1, 2, 0, 1, 0, 1, 1, 1, 1, 0, 1,
       3, 1, 1, 1, 1, 0, 2, 1, 1, 1, 1, 2, 1, 0, 0, 1, 1, 0, 2, 3, 0, 2,
       1, 1, 2, 2, 0, 2, 1, 1, 0, 2, 0, 1, 0, 1, 1, 0, 1, 0, 2, 0, 1, 1,
       2, 1, 2, 1, 1, 2, 0, 1, 2, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 3, 0, 3,
       2, 1, 1, 1, 2, 1, 0, 1, 1, 1, 0, 0, 3, 0, 2, 1, 1, 2, 2, 0, 2, 1,
       2, 0, 2, 0, 1, 0])

In [21]:
import numpy as np
from collections import Counter

def fix_single_sample_classes(X, y_T, y_N, y_pain, min_count=3):
    """
    Ensures each (T, N) class combination has at least `min_count` samples
    by duplicating rows.

    Parameters:
        X       : feature matrix (numpy array)
        y_T     : T_stage labels
        y_N     : N_stage labels
        y_pain  : regression target
        min_count : minimum samples required per class

    Returns:
        X_new, y_T_new, y_N_new, y_pain_new
    """

    X = np.asarray(X)
    y_T = np.asarray(y_T)
    y_N = np.asarray(y_N)
    y_pain = np.asarray(y_pain)

    # 🔥 Combine T and N into joint class
    y_joint = np.array([f"{t}_{n}" for t, n in zip(y_T, y_N)])

    class_counts = Counter(y_joint)

    X_list = [X]
    yT_list = [y_T]
    yN_list = [y_N]
    ypain_list = [y_pain]

    for cls, count in class_counts.items():
        if count < min_count:
            needed = min_count - count

            # indices of this joint class
            idx = np.where(y_joint == cls)[0]

            # duplicate
            dup_idx = np.random.choice(idx, size=needed, replace=True)

            X_list.append(X[dup_idx])
            yT_list.append(y_T[dup_idx])
            yN_list.append(y_N[dup_idx])
            ypain_list.append(y_pain[dup_idx])

    # combine
    X_new = np.vstack(X_list)
    y_T_new = np.concatenate(yT_list)
    y_N_new = np.concatenate(yN_list)
    y_pain_new = np.concatenate(ypain_list)

    return X_new, y_T_new, y_N_new, y_pain_new

In [22]:
# ============================================================
# Phase 1 benchmark code: standard classification + ordinal classification + regression
# Paste this AFTER X1, X2, X3 and y_* arrays are created.
# ============================================================

import numpy as np
import pandas as pd

from copy import deepcopy
from dataclasses import dataclass

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC, SVR
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.linear_model import LogisticRegression, ElasticNet, Ridge
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    explained_variance_score,
)

RANDOM_STATE = 42
TEST_SIZE = 0.20


# -----------------------------
# Ordinal classifier wrapper
# -----------------------------
class OrdinalClassifier(BaseEstimator, ClassifierMixin):
    """
    Threshold-based ordinal classifier.

    For K ordered classes, trains K-1 binary classifiers:
      P(y > 0), P(y > 1), ..., P(y > K-2)

    Prediction is made from cumulative probabilities.
    """

    def __init__(self, base_estimator=None):
        self.base_estimator = base_estimator

    def fit(self, X, y):
        self.le_ = LabelEncoder()
        y_enc = self.le_.fit_transform(np.asarray(y).ravel())
        self.classes_ = self.le_.classes_
        self.n_classes_ = len(self.classes_)

        if self.n_classes_ < 2:
            raise ValueError("OrdinalClassifier needs at least 2 classes.")

        if self.base_estimator is None:
            # safe default for tabular data
            self.base_estimator = make_pipeline(
                StandardScaler(),
                LogisticRegression(
                    solver="saga",
                    penalty="l2",
                    max_iter=5000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            )

        self.models_ = []
        for k in range(self.n_classes_ - 1):
            y_bin = (y_enc > k).astype(int)
            model = clone(self.base_estimator)
            model.fit(X, y_bin)
            self.models_.append(model)

        return self

    def _positive_proba(self, model, X):
        # Prefer predict_proba if available
        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X)
            # positive class probability is column 1
            return proba[:, 1]
        # fallback to decision_function if needed
        if hasattr(model, "decision_function"):
            scores = model.decision_function(X)
            # convert to pseudo-probability
            return 1.0 / (1.0 + np.exp(-scores))
        raise ValueError("Base estimator must support predict_proba or decision_function.")

    def predict_proba(self, X):
        # p_gt[k] = P(y > k)
        p_gt = np.column_stack([self._positive_proba(m, X) for m in self.models_])

        # Convert cumulative probs to class probs
        probs = np.zeros((X.shape[0], self.n_classes_), dtype=float)

        probs[:, 0] = 1.0 - p_gt[:, 0]
        for k in range(1, self.n_classes_ - 1):
            probs[:, k] = p_gt[:, k - 1] - p_gt[:, k]
        probs[:, -1] = p_gt[:, -1]

        probs = np.clip(probs, 1e-12, None)
        probs = probs / probs.sum(axis=1, keepdims=True)
        return probs

    def predict(self, X):
        probs = self.predict_proba(X)
        enc_pred = np.argmax(probs, axis=1)
        return self.le_.inverse_transform(enc_pred)


# -----------------------------
# Metrics helpers
# -----------------------------
def classification_metrics(y_true, y_pred):
    return {
        "acc": accuracy_score(y_true, y_pred),
        "bal_acc": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "cm": confusion_matrix(y_true, y_pred),
    }


def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mse),
        "mse": mse,
        "r2": r2_score(y_true, y_pred),
        "evs": explained_variance_score(y_true, y_pred),
    }


def print_classification_result(name, metrics_dict):
    print(f"\n--- {name} ---")
    print(
        f"Acc: {metrics_dict['acc']:.4f} | "
        f"BalAcc: {metrics_dict['bal_acc']:.4f} | "
        f"MacroF1: {metrics_dict['macro_f1']:.4f} | "
        f"WeightedF1: {metrics_dict['weighted_f1']:.4f}"
    )
    print(
        f"MacroPrecision: {metrics_dict['macro_precision']:.4f} | "
        f"MacroRecall: {metrics_dict['macro_recall']:.4f}"
    )
    print("Confusion matrix:\n", metrics_dict["cm"])


def print_regression_result(name, metrics_dict):
    print(f"\n--- {name} ---")
    print(
        f"MAE: {metrics_dict['mae']:.4f} | "
        f"RMSE: {metrics_dict['rmse']:.4f} | "
        f"MSE: {metrics_dict['mse']:.4f} | "
        f"R2: {metrics_dict['r2']:.4f} | "
        f"EVS: {metrics_dict['evs']:.4f}"
    )


# -----------------------------
# Model factories
# -----------------------------
def get_classification_models(random_state=RANDOM_STATE):
    """
    Non-ordinal classification models.
    ElasticNet here is implemented as LogisticRegression(penalty='elasticnet').
    """
    return {
        "SVM-RBF": make_pipeline(
            StandardScaler(),
            SVC(
                kernel="rbf",
                C=3.0,
                gamma="scale",
                class_weight="balanced",
                probability=True,
                random_state=random_state,
            ),
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=600,
            random_state=random_state,
            n_jobs=-1,
            class_weight="balanced_subsample",
            min_samples_leaf=1,
        ),
        "ElasticNet-LogReg": make_pipeline(
            StandardScaler(),
            LogisticRegression(
                penalty="elasticnet",
                solver="saga",
                l1_ratio=0.5,
                C=1.0,
                max_iter=5000,
                class_weight="balanced",
                random_state=random_state,
            ),
        ),
        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=700,
            random_state=random_state,
            n_jobs=-1,
            class_weight="balanced",
            min_samples_leaf=1,
        ),
    }


def get_ordinal_classification_models(random_state=RANDOM_STATE):
    """
    Ordinal classification models using the OrdinalClassifier wrapper.
    Base estimators are binary classifiers.
    """
    return {
        "Ord-SVM-RBF": OrdinalClassifier(
            base_estimator=make_pipeline(
                StandardScaler(),
                SVC(
                    kernel="rbf",
                    C=3.0,
                    gamma="scale",
                    class_weight="balanced",
                    probability=True,
                    random_state=random_state,
                ),
            )
        ),
        "Ord-RandomForest": OrdinalClassifier(
            base_estimator=RandomForestClassifier(
                n_estimators=600,
                random_state=random_state,
                n_jobs=-1,
                class_weight="balanced_subsample",
                min_samples_leaf=1,
            )
        ),
        "Ord-ElasticNet-LogReg": OrdinalClassifier(
            base_estimator=make_pipeline(
                StandardScaler(),
                LogisticRegression(
                    penalty="elasticnet",
                    solver="saga",
                    l1_ratio=0.5,
                    C=1.0,
                    max_iter=5000,
                    class_weight="balanced",
                    random_state=random_state,
                ),
            )
        ),
        "Ord-ExtraTrees": OrdinalClassifier(
            base_estimator=ExtraTreesClassifier(
                n_estimators=700,
                random_state=random_state,
                n_jobs=-1,
                class_weight="balanced",
                min_samples_leaf=1,
            )
        ),
    }


def get_regression_models(random_state=RANDOM_STATE):
    return {
        "SVR-RBF": make_pipeline(
            StandardScaler(),
            SVR(C=10.0, epsilon=0.05, kernel="rbf", gamma="scale"),
        ),
        "RandomForestRegressor": RandomForestRegressor(
            n_estimators=700,
            random_state=random_state,
            n_jobs=-1,
            min_samples_leaf=1,
        ),
        "ElasticNet": make_pipeline(
            StandardScaler(),
            ElasticNet(
                alpha=0.01,
                l1_ratio=0.5,
                max_iter=20000,
                random_state=random_state,
            ),
        ),
        "Ridge": make_pipeline(
            StandardScaler(),
            Ridge(alpha=1.0, random_state=random_state),
        ),
        "ExtraTreesRegressor": ExtraTreesRegressor(
            n_estimators=700,
            random_state=random_state,
            n_jobs=-1,
            min_samples_leaf=1,
        ),
    }


# -----------------------------
# Generic runner
# -----------------------------
def run_classification_suite(X, y, dataset_name, target_name, ordinal=False,
                             test_size=TEST_SIZE, random_state=RANDOM_STATE):
    """
    Runs either standard or ordinal classification models on a single target.
    """
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y).ravel()

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    print(f"\n{'='*20} {dataset_name.upper()} | {target_name.upper()} | "
          f"{'ORDINAL' if ordinal else 'STANDARD'} {'='*20}")
    print(f"Train shape: {X_train.shape} | Test shape: {X_test.shape}")
    print("Train class counts:", dict(pd.Series(y_train).value_counts().sort_index()))
    print("Test class counts :", dict(pd.Series(y_test).value_counts().sort_index()))

    models = get_ordinal_classification_models(random_state) if ordinal else get_classification_models(random_state)

    rows = []
    for model_name, model in models.items():
        try:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            m = classification_metrics(y_test, y_pred)
            print_classification_result(f"{dataset_name} | {target_name} | {model_name}", m)

            rows.append({
                "dataset": dataset_name,
                "target": target_name,
                "setting": "ordinal" if ordinal else "standard",
                "model": model_name,
                "acc": m["acc"],
                "bal_acc": m["bal_acc"],
                "macro_f1": m["macro_f1"],
                "weighted_f1": m["weighted_f1"],
                "macro_precision": m["macro_precision"],
                "macro_recall": m["macro_recall"],
            })
        except Exception as e:
            print(f"\n--- {dataset_name} | {target_name} | {model_name} FAILED ---")
            print(type(e).__name__ + ":", e)

    return pd.DataFrame(rows)


def run_regression_suite(X, y, dataset_name, target_name="pain",
                         test_size=TEST_SIZE, random_state=RANDOM_STATE):
    """
    Runs regression models on pain.
    """
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32).ravel()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state
    )

    print(f"\n{'='*20} {dataset_name.upper()} | {target_name.upper()} | REGRESSION {'='*20}")
    print(f"Train shape: {X_train.shape} | Test shape: {X_test.shape}")

    models = get_regression_models(random_state)

    rows = []
    for model_name, model in models.items():
        try:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            m = regression_metrics(y_test, y_pred)
            print_regression_result(f"{dataset_name} | {target_name} | {model_name}", m)

            rows.append({
                "dataset": dataset_name,
                "target": target_name,
                "model": model_name,
                "mae": m["mae"],
                "rmse": m["rmse"],
                "mse": m["mse"],
                "r2": m["r2"],
                "evs": m["evs"],
            })
        except Exception as e:
            print(f"\n--- {dataset_name} | {target_name} | {model_name} FAILED ---")
            print(type(e).__name__ + ":", e)

    return pd.DataFrame(rows)


def run_full_benchmark(X, y_T, y_N, y_pain, dataset_name):
    """
    Runs:
      1) standard classification on T and N
      2) ordinal classification on T and N
      3) regression on pain
    """
    results = {}

    # Standard classification
    res_T_std = run_classification_suite(X, y_T, dataset_name, "T", ordinal=False)
    res_N_std = run_classification_suite(X, y_N, dataset_name, "N", ordinal=False)

    # Ordinal classification
    res_T_ord = run_classification_suite(X, y_T, dataset_name, "T", ordinal=True)
    res_N_ord = run_classification_suite(X, y_N, dataset_name, "N", ordinal=True)

    # Regression
    res_pain = run_regression_suite(X, y_pain, dataset_name, target_name="pain")

    results["T_standard"] = res_T_std
    results["N_standard"] = res_N_std
    results["T_ordinal"] = res_T_ord
    results["N_ordinal"] = res_N_ord
    results["pain_regression"] = res_pain

    return results


# ============================================================
# Example runs for your three feature sets
# ============================================================
# Assumes you already have:
# X1, y_T1, y_N1, y_pain1   -> value features
# X2, y_T2, y_N2, y_pain2   -> vpk features
# X3, y_T3, y_N3, y_pain3   -> all features
#
# Uncomment the blocks below to run.

results_value = run_full_benchmark(X1, y_T1, y_N1, y_pain1, dataset_name="value")
results_vpk   = run_full_benchmark(X2, y_T2, y_N2, y_pain2, dataset_name="vpk")
results_all   = run_full_benchmark(X3, y_T3, y_N3, y_pain3, dataset_name="all")

# ------------------------------------------------------------
# Summary tables
# ------------------------------------------------------------
summary_classification = pd.concat([
    results_value["T_standard"], results_value["N_standard"],
    results_value["T_ordinal"], results_value["N_ordinal"],
    results_vpk["T_standard"], results_vpk["N_standard"],
    results_vpk["T_ordinal"], results_vpk["N_ordinal"],
    results_all["T_standard"], results_all["N_standard"],
    results_all["T_ordinal"], results_all["N_ordinal"],
], ignore_index=True)

summary_regression = pd.concat([
    results_value["pain_regression"],
    results_vpk["pain_regression"],
    results_all["pain_regression"],
], ignore_index=True)

print("\n\n================ CLASSIFICATION SUMMARY ================\n")
print(summary_classification.sort_values(["dataset", "target", "setting", "bal_acc"], ascending=[True, True, True, False]).to_string(index=False))

print("\n\n================ REGRESSION SUMMARY ================\n")
print(summary_regression.sort_values(["dataset", "r2"], ascending=[True, False]).to_string(index=False))


==================== VALUE | T | STANDARD ====================
Train shape: (180, 54) | Test shape: (46, 54)
Train class counts: {0: 4, 1: 28, 2: 106, 3: 42}
Test class counts : {0: 1, 1: 7, 2: 27, 3: 11}

--- value | T | SVM-RBF ---
Acc: 0.3043 | BalAcc: 0.2359 | MacroF1: 0.2166 | WeightedF1: 0.3166
MacroPrecision: 0.2250 | MacroRecall: 0.2359
Confusion matrix:
 [[ 0  0  1  0]
 [ 0  3  4  0]
 [ 2 10  9  6]
 [ 1  2  6  2]]

--- value | T | RandomForest ---
Acc: 0.4348 | BalAcc: 0.2251 | MacroF1: 0.2186 | WeightedF1: 0.3999
MacroPrecision: 0.2240 | MacroRecall: 0.2251
Confusion matrix:
 [[ 0  0  1  0]
 [ 0  1  5  1]
 [ 1  4 18  4]
 [ 0  0 10  1]]

--- value | T | ElasticNet-LogReg ---
Acc: 0.2609 | BalAcc: 0.2174 | MacroF1: 0.1893 | WeightedF1: 0.2753
MacroPrecision: 0.2025 | MacroRecall: 0.2174
Confusion matrix:
 [[ 0  0  0  1]
 [ 0  3  3  1]
 [ 2 10  7  8]
 [ 0  4  5  2]]

--- value | T | ExtraTrees ---
Acc: 0.4348 | BalAcc: 0.2386 | MacroF1: 0.2386 | WeightedF1: 0.4163
MacroPrecisio

/Users/raghavakarthikeyakadaveru/Desktop/Raghava/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



--- vpk | N | ElasticNet-LogReg ---
Acc: 0.3261 | BalAcc: 0.2619 | MacroF1: 0.2433 | WeightedF1: 0.3202
MacroPrecision: 0.2412 | MacroRecall: 0.2619
Confusion matrix:
 [[2 5 4 1]
 [5 8 8 0]
 [2 2 5 1]
 [1 2 0 0]]

--- vpk | N | ExtraTrees ---
Acc: 0.2826 | BalAcc: 0.2250 | MacroF1: 0.2167 | WeightedF1: 0.2812
MacroPrecision: 0.2126 | MacroRecall: 0.2250
Confusion matrix:
 [[2 6 3 1]
 [7 7 6 1]
 [3 3 4 0]
 [1 2 0 0]]

==================== VPK | T | ORDINAL ====================
Train shape: (180, 220) | Test shape: (46, 220)
Train class counts: {0: 4, 1: 28, 2: 106, 3: 42}
Test class counts : {0: 1, 1: 7, 2: 27, 3: 11}

--- vpk | T | Ord-SVM-RBF ---
Acc: 0.5870 | BalAcc: 0.2500 | MacroF1: 0.1849 | WeightedF1: 0.4342
MacroPrecision: 0.1467 | MacroRecall: 0.2500
Confusion matrix:
 [[ 0  0  1  0]
 [ 0  0  7  0]
 [ 0  0 27  0]
 [ 0  0 11  0]]

--- vpk | T | Ord-RandomForest ---
Acc: 0.5870 | BalAcc: 0.3438 | MacroF1: 0.3416 | WeightedF1: 0.5570
MacroPrecision: 0.3610 | MacroRecall: 0.3438
C

/Users/raghavakarthikeyakadaveru/Desktop/Raghava/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



--- vpk | N | Ord-ElasticNet-LogReg ---
Acc: 0.2609 | BalAcc: 0.2137 | MacroF1: 0.2091 | WeightedF1: 0.2747
MacroPrecision: 0.2484 | MacroRecall: 0.2137
Confusion matrix:
 [[5 3 2 2]
 [8 5 6 2]
 [6 0 2 2]
 [2 1 0 0]]

--- vpk | N | Ord-ExtraTrees ---
Acc: 0.3261 | BalAcc: 0.2488 | MacroF1: 0.2450 | WeightedF1: 0.3250
MacroPrecision: 0.2419 | MacroRecall: 0.2488
Confusion matrix:
 [[2 6 3 1]
 [7 9 4 1]
 [3 3 4 0]
 [1 2 0 0]]

==================== VPK | PAIN | REGRESSION ====================
Train shape: (180, 220) | Test shape: (46, 220)

--- vpk | pain | SVR-RBF ---
MAE: 0.2223 | RMSE: 0.2860 | MSE: 0.0818 | R2: -0.6010 | EVS: -0.5452

--- vpk | pain | RandomForestRegressor ---
MAE: 0.2066 | RMSE: 0.2591 | MSE: 0.0671 | R2: -0.3139 | EVS: -0.1803

--- vpk | pain | ElasticNet ---
MAE: 0.1996 | RMSE: 0.2428 | MSE: 0.0590 | R2: -0.1541 | EVS: 0.0013

--- vpk | pain | Ridge ---
MAE: 0.2138 | RMSE: 0.2515 | MSE: 0.0632 | R2: -0.2374 | EVS: -0.0903

--- vpk | pain | ExtraTreesRegressor ---


/Users/raghavakarthikeyakadaveru/Desktop/Raghava/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



--- all | N | ElasticNet-LogReg ---
Acc: 0.2609 | BalAcc: 0.2089 | MacroF1: 0.2058 | WeightedF1: 0.2691
MacroPrecision: 0.2113 | MacroRecall: 0.2089
Confusion matrix:
 [[3 4 4 1]
 [6 6 7 2]
 [3 3 3 1]
 [1 2 0 0]]

--- all | N | ExtraTrees ---
Acc: 0.4565 | BalAcc: 0.3119 | MacroF1: 0.2988 | WeightedF1: 0.4232
MacroPrecision: 0.2991 | MacroRecall: 0.3119
Confusion matrix:
 [[ 4  6  2  0]
 [ 4 15  2  0]
 [ 3  5  2  0]
 [ 2  1  0  0]]

==================== ALL | T | ORDINAL ====================
Train shape: (180, 256) | Test shape: (46, 256)
Train class counts: {0: 4, 1: 28, 2: 106, 3: 42}
Test class counts : {0: 1, 1: 7, 2: 27, 3: 11}

--- all | T | Ord-SVM-RBF ---
Acc: 0.5870 | BalAcc: 0.2500 | MacroF1: 0.1849 | WeightedF1: 0.4342
MacroPrecision: 0.1467 | MacroRecall: 0.2500
Confusion matrix:
 [[ 0  0  1  0]
 [ 0  0  7  0]
 [ 0  0 27  0]
 [ 0  0 11  0]]

--- all | T | Ord-RandomForest ---
Acc: 0.5217 | BalAcc: 0.2357 | MacroF1: 0.2029 | WeightedF1: 0.4329
MacroPrecision: 0.1937 | Macro

/Users/raghavakarthikeyakadaveru/Desktop/Raghava/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



--- all | N | Ord-ElasticNet-LogReg ---
Acc: 0.3913 | BalAcc: 0.3292 | MacroF1: 0.3063 | WeightedF1: 0.3869
MacroPrecision: 0.3167 | MacroRecall: 0.3292
Confusion matrix:
 [[7 2 2 1]
 [8 7 6 0]
 [3 2 4 1]
 [2 1 0 0]]

--- all | N | Ord-ExtraTrees ---
Acc: 0.4565 | BalAcc: 0.3119 | MacroF1: 0.2988 | WeightedF1: 0.4292
MacroPrecision: 0.2929 | MacroRecall: 0.3119
Confusion matrix:
 [[ 4  5  3  0]
 [ 4 15  2  0]
 [ 5  3  2  0]
 [ 1  2  0  0]]

==================== ALL | PAIN | REGRESSION ====================
Train shape: (180, 256) | Test shape: (46, 256)

--- all | pain | SVR-RBF ---
MAE: 0.1288 | RMSE: 0.1599 | MSE: 0.0256 | R2: 0.4996 | EVS: 0.5561

--- all | pain | RandomForestRegressor ---
MAE: 0.1129 | RMSE: 0.1414 | MSE: 0.0200 | R2: 0.6086 | EVS: 0.6391

--- all | pain | ElasticNet ---
MAE: 0.1056 | RMSE: 0.1343 | MSE: 0.0180 | R2: 0.6471 | EVS: 0.6543

--- all | pain | Ridge ---
MAE: 0.1038 | RMSE: 0.1298 | MSE: 0.0169 | R2: 0.6701 | EVS: 0.6772

--- all | pain | ExtraTreesRegre

In [23]:
import numpy as np
import pandas as pd

from copy import deepcopy
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression, Ridge, ElasticNet
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, confusion_matrix
)
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression

RANDOM_STATE = 42


# =========================
# Helpers
# =========================
class OrdinalClassifier(BaseEstimator, ClassifierMixin):
    """
    Threshold-based ordinal classifier.
    Trains K-1 binary classifiers for K ordered classes.
    """
    def __init__(self, base_estimator=None):
        self.base_estimator = base_estimator

    def fit(self, X, y):
        self.le_ = LabelEncoder()
        y_enc = self.le_.fit_transform(np.asarray(y).ravel())
        self.classes_ = self.le_.classes_
        self.n_classes_ = len(self.classes_)

        if self.n_classes_ < 2:
            raise ValueError("Need at least 2 classes.")

        if self.base_estimator is None:
            self.base_estimator = make_pipeline(
                StandardScaler(),
                LogisticRegression(
                    solver="saga",
                    penalty="l2",
                    max_iter=5000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            )

        self.models_ = []
        for k in range(self.n_classes_ - 1):
            y_bin = (y_enc > k).astype(int)
            model = clone(self.base_estimator)
            model.fit(X, y_bin)
            self.models_.append(model)
        return self

    def _positive_proba(self, model, X):
        if hasattr(model, "predict_proba"):
            return model.predict_proba(X)[:, 1]
        if hasattr(model, "decision_function"):
            s = model.decision_function(X)
            return 1.0 / (1.0 + np.exp(-s))
        raise ValueError("Base estimator must support predict_proba or decision_function.")

    def predict_proba(self, X):
        p_gt = np.column_stack([self._positive_proba(m, X) for m in self.models_])

        probs = np.zeros((X.shape[0], self.n_classes_), dtype=float)
        probs[:, 0] = 1.0 - p_gt[:, 0]
        for k in range(1, self.n_classes_ - 1):
            probs[:, k] = p_gt[:, k - 1] - p_gt[:, k]
        probs[:, -1] = p_gt[:, -1]

        probs = np.clip(probs, 1e-12, None)
        probs /= probs.sum(axis=1, keepdims=True)
        return probs

    def predict(self, X):
        probs = self.predict_proba(X)
        enc_pred = np.argmax(probs, axis=1)
        return self.le_.inverse_transform(enc_pred)


def oversample_to_min_count(X, y, class_to_min_count=None, random_state=42):
    """
    Oversample only within the training fold.
    Example: class_to_min_count={0: 10} duplicates class 0 until it has at least 10 samples.
    """
    if class_to_min_count is None:
        return X, y

    rng = np.random.default_rng(random_state)
    X = np.asarray(X)
    y = np.asarray(y)

    X_parts = [X]
    y_parts = [y]

    for cls, min_count in class_to_min_count.items():
        idx = np.where(y == cls)[0]
        if len(idx) == 0:
            continue
        if len(idx) >= min_count:
            continue
        extra = rng.choice(idx, size=min_count - len(idx), replace=True)
        X_parts.append(X[extra])
        y_parts.append(y[extra])

    X_new = np.vstack(X_parts)
    y_new = np.concatenate(y_parts)

    perm = rng.permutation(len(y_new))
    return X_new[perm], y_new[perm]


def classification_metrics(y_true, y_pred):
    return {
        "acc": accuracy_score(y_true, y_pred),
        "bal_acc": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "cm": confusion_matrix(y_true, y_pred),
    }


def print_fold_summary(name, m):
    print(f"\n--- {name} ---")
    print(
        f"Acc: {m['acc']:.4f} | BalAcc: {m['bal_acc']:.4f} | "
        f"MacroF1: {m['macro_f1']:.4f} | WeightedF1: {m['weighted_f1']:.4f}"
    )
    print(
        f"MacroPrecision: {m['macro_precision']:.4f} | "
        f"MacroRecall: {m['macro_recall']:.4f}"
    )
    print("Confusion matrix:\n", m["cm"])


def summarize_cv_rows(df):
    agg = (
        df.groupby(["dataset", "target", "setting", "model"], as_index=False)
          .agg(
              acc_mean=("acc", "mean"),
              acc_std=("acc", "std"),
              bal_acc_mean=("bal_acc", "mean"),
              bal_acc_std=("bal_acc", "std"),
              macro_f1_mean=("macro_f1", "mean"),
              macro_f1_std=("macro_f1", "std"),
              weighted_f1_mean=("weighted_f1", "mean"),
              weighted_f1_std=("weighted_f1", "std"),
              macro_precision_mean=("macro_precision", "mean"),
              macro_recall_mean=("macro_recall", "mean"),
          )
    )
    return agg.sort_values(["dataset", "target", "setting", "bal_acc_mean"], ascending=[True, True, True, False])


def compute_thresholds_from_train_scores(train_scores, y_train_enc, n_classes):
    means = []
    for c in range(n_classes):
        vals = train_scores[y_train_enc == c]
        means.append(np.mean(vals) if len(vals) > 0 else np.nan)

    if np.any(np.isnan(means)):
        # fallback if a class disappears inside a fold for any reason
        qs = [i / n_classes for i in range(1, n_classes)]
        return np.quantile(train_scores, qs)

    thresholds = [(means[i] + means[i + 1]) / 2.0 for i in range(n_classes - 1)]
    return np.asarray(thresholds, dtype=float)


def latent_regression_predict(model, X_train, y_train_enc, X_test, mode="threshold"):
    model.fit(X_train, y_train_enc)
    train_scores = model.predict(X_train)
    test_scores = model.predict(X_test)
    n_classes = len(np.unique(y_train_enc))

    if mode == "round":
        pred = np.rint(test_scores).astype(int)
        pred = np.clip(pred, 0, n_classes - 1)
        return pred

    thresholds = compute_thresholds_from_train_scores(train_scores, y_train_enc, n_classes)
    pred = np.digitize(test_scores, thresholds)
    pred = np.clip(pred, 0, n_classes - 1)
    return pred


# =========================
# Model pools
# =========================
def get_standard_models():
    return {
        "DummyMostFreq": DummyClassifier(strategy="most_frequent"),
        "DummyStratified": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
        "SVM-RBF": make_pipeline(
            StandardScaler(),
            SVC(
                kernel="rbf",
                C=3.0,
                gamma="scale",
                class_weight="balanced",
                probability=True,
                random_state=RANDOM_STATE,
            ),
        ),
        "ElasticNet-LogReg": make_pipeline(
            StandardScaler(),
            LogisticRegression(
                penalty="elasticnet",
                solver="saga",
                l1_ratio=0.5,
                C=1.0,
                max_iter=5000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=700,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced_subsample",
        ),
        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=700,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced",
        ),
    }


def get_ordinal_models():
    return {
        "Ord-SVM-RBF": OrdinalClassifier(
            base_estimator=make_pipeline(
                StandardScaler(),
                SVC(
                    kernel="rbf",
                    C=3.0,
                    gamma="scale",
                    class_weight="balanced",
                    probability=True,
                    random_state=RANDOM_STATE,
                ),
            )
        ),
        "Ord-ElasticNet-LogReg": OrdinalClassifier(
            base_estimator=make_pipeline(
                StandardScaler(),
                LogisticRegression(
                    penalty="elasticnet",
                    solver="saga",
                    l1_ratio=0.5,
                    C=1.0,
                    max_iter=5000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            )
        ),
        "Ord-RandomForest": OrdinalClassifier(
            base_estimator=RandomForestClassifier(
                n_estimators=700,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                class_weight="balanced_subsample",
            )
        ),
        "Ord-ExtraTrees": OrdinalClassifier(
            base_estimator=ExtraTreesClassifier(
                n_estimators=700,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                class_weight="balanced",
            )
        ),
    }


def get_latent_regressors():
    return {
        "Ridge-round": Ridge(alpha=1.0, random_state=RANDOM_STATE),
        "ElasticNet-round": make_pipeline(
            StandardScaler(),
            ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=20000, random_state=RANDOM_STATE),
        ),
        "SVR-round": make_pipeline(
            StandardScaler(),
            SVR(C=10.0, epsilon=0.05, kernel="rbf", gamma="scale"),
        ),
        "RFReg-round": RandomForestRegressor(
            n_estimators=700,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "ETReg-round": ExtraTreesRegressor(
            n_estimators=700,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "Ridge-threshold": Ridge(alpha=1.0, random_state=RANDOM_STATE),
        "ElasticNet-threshold": make_pipeline(
            StandardScaler(),
            ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=20000, random_state=RANDOM_STATE),
        ),
        "SVR-threshold": make_pipeline(
            StandardScaler(),
            SVR(C=10.0, epsilon=0.05, kernel="rbf", gamma="scale"),
        ),
        "RFReg-threshold": RandomForestRegressor(
            n_estimators=700,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "ETReg-threshold": ExtraTreesRegressor(
            n_estimators=700,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    }


# =========================
# CV runners
# =========================
def run_standard_cv(X, y, dataset_name, target_name, class_to_min_count=None, n_splits=5, n_repeats=3):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y).ravel()

    rkf = RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=n_repeats,
        random_state=RANDOM_STATE,
    )

    models = get_standard_models()
    rows = []

    print(f"\n{'='*20} {dataset_name.upper()} | {target_name.upper()} | STANDARD {'='*20}")
    print("Class counts:", dict(pd.Series(y).value_counts().sort_index()))

    for model_name, model in models.items():
        fold_scores = []
        for fold, (tr_idx, te_idx) in enumerate(rkf.split(X, y), start=1):
            X_tr, X_te = X[tr_idx], X[te_idx]
            y_tr, y_te = y[tr_idx], y[te_idx]

            if class_to_min_count is not None:
                X_tr, y_tr = oversample_to_min_count(X_tr, y_tr, class_to_min_count=class_to_min_count)

            try:
                model_fit = clone(model)
                model_fit.fit(X_tr, y_tr)
                y_pred = model_fit.predict(X_te)
                m = classification_metrics(y_te, y_pred)
                fold_scores.append(m)
            except Exception as e:
                print(f"[{dataset_name}-{target_name}-{model_name}] fold {fold} failed: {type(e).__name__}: {e}")

        if len(fold_scores) == 0:
            continue

        avg = {
            "dataset": dataset_name,
            "target": target_name,
            "setting": "standard",
            "model": model_name,
            "acc": np.mean([m["acc"] for m in fold_scores]),
            "bal_acc": np.mean([m["bal_acc"] for m in fold_scores]),
            "macro_f1": np.mean([m["macro_f1"] for m in fold_scores]),
            "weighted_f1": np.mean([m["weighted_f1"] for m in fold_scores]),
            "macro_precision": np.mean([m["macro_precision"] for m in fold_scores]),
            "macro_recall": np.mean([m["macro_recall"] for m in fold_scores]),
        }
        rows.append(avg)

        print_fold_summary(f"{dataset_name} | {target_name} | {model_name}", fold_scores[-1])

    return pd.DataFrame(rows)


def run_ordinal_cv(X, y, dataset_name, target_name, class_to_min_count=None, n_splits=5, n_repeats=3):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y).ravel()

    rkf = RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=n_repeats,
        random_state=RANDOM_STATE,
    )

    models = get_ordinal_models()
    rows = []

    print(f"\n{'='*20} {dataset_name.upper()} | {target_name.upper()} | ORDINAL {'='*20}")
    print("Class counts:", dict(pd.Series(y).value_counts().sort_index()))

    for model_name, model in models.items():
        fold_scores = []
        for fold, (tr_idx, te_idx) in enumerate(rkf.split(X, y), start=1):
            X_tr, X_te = X[tr_idx], X[te_idx]
            y_tr, y_te = y[tr_idx], y[te_idx]

            if class_to_min_count is not None:
                X_tr, y_tr = oversample_to_min_count(X_tr, y_tr, class_to_min_count=class_to_min_count)

            try:
                model_fit = clone(model)
                model_fit.fit(X_tr, y_tr)
                y_pred = model_fit.predict(X_te)
                m = classification_metrics(y_te, y_pred)
                fold_scores.append(m)
            except Exception as e:
                print(f"[{dataset_name}-{target_name}-{model_name}] fold {fold} failed: {type(e).__name__}: {e}")

        if len(fold_scores) == 0:
            continue

        avg = {
            "dataset": dataset_name,
            "target": target_name,
            "setting": "ordinal",
            "model": model_name,
            "acc": np.mean([m["acc"] for m in fold_scores]),
            "bal_acc": np.mean([m["bal_acc"] for m in fold_scores]),
            "macro_f1": np.mean([m["macro_f1"] for m in fold_scores]),
            "weighted_f1": np.mean([m["weighted_f1"] for m in fold_scores]),
            "macro_precision": np.mean([m["macro_precision"] for m in fold_scores]),
            "macro_recall": np.mean([m["macro_recall"] for m in fold_scores]),
        }
        rows.append(avg)

        print_fold_summary(f"{dataset_name} | {target_name} | {model_name}", fold_scores[-1])

    return pd.DataFrame(rows)


def run_latent_cv(X, y, dataset_name, target_name, class_to_min_count=None, n_splits=5, n_repeats=3):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y).ravel().astype(int)

    rkf = RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=n_repeats,
        random_state=RANDOM_STATE,
    )

    models = get_latent_regressors()
    rows = []

    print(f"\n{'='*20} {dataset_name.upper()} | {target_name.upper()} | LATENT REGRESSION {'='*20}")
    print("Class counts:", dict(pd.Series(y).value_counts().sort_index()))

    for model_name, model in models.items():
        mode = "round" if "round" in model_name else "threshold"
        base_name = model_name.replace("-round", "").replace("-threshold", "")

        fold_scores = []
        for fold, (tr_idx, te_idx) in enumerate(rkf.split(X, y), start=1):
            X_tr, X_te = X[tr_idx], X[te_idx]
            y_tr, y_te = y[tr_idx], y[te_idx]

            if class_to_min_count is not None:
                X_tr, y_tr = oversample_to_min_count(X_tr, y_tr, class_to_min_count=class_to_min_count)

            try:
                # regressors need encoded labels starting at 0
                le = LabelEncoder()
                y_tr_enc = le.fit_transform(y_tr)
                y_te_enc = le.transform(y_te)

                model_fit = clone(model)
                y_pred_enc = latent_regression_predict(
                    model_fit, X_tr, y_tr_enc, X_te, mode=mode
                )
                y_pred = le.inverse_transform(y_pred_enc)

                m = classification_metrics(y_te, y_pred)
                fold_scores.append(m)
            except Exception as e:
                print(f"[{dataset_name}-{target_name}-{model_name}] fold {fold} failed: {type(e).__name__}: {e}")

        if len(fold_scores) == 0:
            continue

        avg = {
            "dataset": dataset_name,
            "target": target_name,
            "setting": "latent_" + mode,
            "model": base_name,
            "acc": np.mean([m["acc"] for m in fold_scores]),
            "bal_acc": np.mean([m["bal_acc"] for m in fold_scores]),
            "macro_f1": np.mean([m["macro_f1"] for m in fold_scores]),
            "weighted_f1": np.mean([m["weighted_f1"] for m in fold_scores]),
            "macro_precision": np.mean([m["macro_precision"] for m in fold_scores]),
            "macro_recall": np.mean([m["macro_recall"] for m in fold_scores]),
        }
        rows.append(avg)

        print_fold_summary(f"{dataset_name} | {target_name} | {model_name}", fold_scores[-1])

    return pd.DataFrame(rows)


# =========================
# Run the requested settings
# =========================
# T: vpk only
# N: all only
#
# Replace these with your actual variables if names differ:
# X2 -> vpk features
# X3 -> all features
# y_T2, y_N3 -> labels for the same rows

results_T_std = run_standard_cv(X2, y_T2, dataset_name="vpk", target_name="T", class_to_min_count={0: 10})
results_T_ord = run_ordinal_cv(X2, y_T2, dataset_name="vpk", target_name="T", class_to_min_count={0: 10})
results_T_lat = run_latent_cv(X2, y_T2, dataset_name="vpk", target_name="T", class_to_min_count={0: 10})

results_N_std = run_standard_cv(X3, y_N3, dataset_name="all", target_name="N", class_to_min_count=None)
results_N_ord = run_ordinal_cv(X3, y_N3, dataset_name="all", target_name="N", class_to_min_count=None)
results_N_lat = run_latent_cv(X3, y_N3, dataset_name="all", target_name="N", class_to_min_count=None)

summary_phase2 = pd.concat(
    [results_T_std, results_T_ord, results_T_lat, results_N_std, results_N_ord, results_N_lat],
    ignore_index=True
)

summary_phase2_sorted = summarize_cv_rows(summary_phase2)

print("\n\n================ PHASE 2 SUMMARY ================\n")
print(summary_phase2_sorted.to_string(index=False))

# Quick signal check for N: compare best model to dummy most-frequent baseline
n_rows = summary_phase2_sorted[summary_phase2_sorted["target"] == "N"].copy()
dummy_n = n_rows[n_rows["model"].isin(["DummyMostFreq", "DummyStratified"])]
best_n = n_rows.sort_values("bal_acc_mean", ascending=False).head(1)

print("\n\n================ N SIGNAL CHECK ================\n")
if len(dummy_n) > 0:
    print("Dummy baselines:")
    print(dummy_n[["setting", "model", "bal_acc_mean", "macro_f1_mean", "acc_mean"]].to_string(index=False))
print("\nBest N model:")
print(best_n[["setting", "model", "bal_acc_mean", "macro_f1_mean", "acc_mean"]].to_string(index=False))


==================== VPK | T | STANDARD ====================
Class counts: {0: 5, 1: 35, 2: 133, 3: 53}

--- vpk | T | DummyMostFreq ---
Acc: 0.5778 | BalAcc: 0.2500 | MacroF1: 0.1831 | WeightedF1: 0.4232
MacroPrecision: 0.1444 | MacroRecall: 0.2500
Confusion matrix:
 [[ 0  0  1  0]
 [ 0  0  7  0]
 [ 0  0 26  0]
 [ 0  0 11  0]]

--- vpk | T | DummyStratified ---
Acc: 0.3778 | BalAcc: 0.2028 | MacroF1: 0.2054 | WeightedF1: 0.3839
MacroPrecision: 0.2082 | MacroRecall: 0.2028
Confusion matrix:
 [[ 0  0  1  0]
 [ 1  0  4  2]
 [ 2  4 14  6]
 [ 1  1  6  3]]

--- vpk | T | SVM-RBF ---
Acc: 0.3778 | BalAcc: 0.2420 | MacroF1: 0.2379 | WeightedF1: 0.3910
MacroPrecision: 0.2408 | MacroRecall: 0.2420
Confusion matrix:
 [[ 0  0  1  0]
 [ 0  1  3  3]
 [ 1  5 12  8]
 [ 0  2  5  4]]

--- vpk | T | ElasticNet-LogReg ---
Acc: 0.3556 | BalAcc: 0.2324 | MacroF1: 0.2311 | WeightedF1: 0.3872
MacroPrecision: 0.2493 | MacroRecall: 0.2324
Confusion matrix:
 [[ 0  1  0  0]
 [ 0  1  4  2]
 [ 0  6 11  9]
 [ 1  4

/Users/raghavakarthikeyakadaveru/Desktop/Raghava/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/raghavakarthikeyakadaveru/Desktop/Raghava/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



--- all | N | ElasticNet-LogReg ---
Acc: 0.3111 | BalAcc: 0.3036 | MacroF1: 0.2745 | WeightedF1: 0.3118
MacroPrecision: 0.2815 | MacroRecall: 0.3036
Confusion matrix:
 [[2 8 2 0]
 [3 8 6 4]
 [0 4 3 2]
 [1 0 1 1]]

--- all | N | RandomForest ---
Acc: 0.4889 | BalAcc: 0.3026 | MacroF1: 0.2682 | WeightedF1: 0.3958
MacroPrecision: 0.3500 | MacroRecall: 0.3026
Confusion matrix:
 [[ 1  9  2  0]
 [ 1 19  1  0]
 [ 0  7  2  0]
 [ 0  3  0  0]]

--- all | N | ExtraTrees ---
Acc: 0.4444 | BalAcc: 0.2629 | MacroF1: 0.2333 | WeightedF1: 0.3696
MacroPrecision: 0.2574 | MacroRecall: 0.2629
Confusion matrix:
 [[ 1  8  2  1]
 [ 2 18  1  0]
 [ 0  6  1  2]
 [ 1  2  0  0]]

==================== ALL | N | ORDINAL ====================
Class counts: {0: 58, 1: 106, 2: 48, 3: 14}

--- all | N | Ord-SVM-RBF ---
Acc: 0.4667 | BalAcc: 0.2500 | MacroF1: 0.1591 | WeightedF1: 0.2970
MacroPrecision: 0.1167 | MacroRecall: 0.2500
Confusion matrix:
 [[ 0 12  0  0]
 [ 0 21  0  0]
 [ 0  9  0  0]
 [ 0  3  0  0]]

--- all 

In [24]:
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

def make_pca_transformer(n_components):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n_components, random_state=RANDOM_STATE)),
    ])


def run_pca_cv(X, y, dataset_name, target_name, setting_type, n_components, class_to_min_count=None, n_splits=5, n_repeats=3):
    """
    PCA version of the previous CV.
    setting_type in {"standard", "ordinal", "latent"}.
    """
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y).ravel()

    rkf = RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=n_repeats,
        random_state=RANDOM_STATE,
    )

    if setting_type == "standard":
        models = get_standard_models()
    elif setting_type == "ordinal":
        models = get_ordinal_models()
    elif setting_type == "latent":
        models = get_latent_regressors()
    else:
        raise ValueError("setting_type must be one of: standard, ordinal, latent")

    rows = []
    print(f"\n{'='*20} {dataset_name.upper()} | {target_name.upper()} | PCA={n_components} | {setting_type.upper()} {'='*20}")

    for model_name, model in models.items():
        if setting_type == "latent":
            mode = "round" if "round" in model_name else "threshold"
            base_name = model_name.replace("-round", "").replace("-threshold", "")
        else:
            base_name = model_name

        fold_scores = []
        for fold, (tr_idx, te_idx) in enumerate(rkf.split(X, y), start=1):
            X_tr, X_te = X[tr_idx], X[te_idx]
            y_tr, y_te = y[tr_idx], y[te_idx]

            if class_to_min_count is not None:
                X_tr, y_tr = oversample_to_min_count(X_tr, y_tr, class_to_min_count=class_to_min_count)

            try:
                # Fit PCA only on the training fold
                pca_pipe = make_pca_transformer(n_components)
                X_tr_p = pca_pipe.fit_transform(X_tr)
                X_te_p = pca_pipe.transform(X_te)

                if setting_type in ("standard", "ordinal"):
                    model_fit = clone(model)
                    model_fit.fit(X_tr_p, y_tr)
                    y_pred = model_fit.predict(X_te_p)

                else:
                    le = LabelEncoder()
                    y_tr_enc = le.fit_transform(y_tr)
                    le.fit(y_tr)
                    model_fit = clone(model)
                    y_pred_enc = latent_regression_predict(
                        model_fit, X_tr_p, y_tr_enc, X_te_p, mode=mode
                    )
                    y_pred = le.inverse_transform(y_pred_enc)

                m = classification_metrics(y_te, y_pred)
                fold_scores.append(m)

            except Exception as e:
                print(f"[{dataset_name}-{target_name}-PCA-{n_components}-{model_name}] fold {fold} failed: {type(e).__name__}: {e}")

        if len(fold_scores) == 0:
            continue

        rows.append({
            "dataset": dataset_name,
            "target": target_name,
            "setting": f"pca_{setting_type}_{n_components}",
            "model": base_name,
            "acc": np.mean([m["acc"] for m in fold_scores]),
            "bal_acc": np.mean([m["bal_acc"] for m in fold_scores]),
            "macro_f1": np.mean([m["macro_f1"] for m in fold_scores]),
            "weighted_f1": np.mean([m["weighted_f1"] for m in fold_scores]),
            "macro_precision": np.mean([m["macro_precision"] for m in fold_scores]),
            "macro_recall": np.mean([m["macro_recall"] for m in fold_scores]),
        })

    return pd.DataFrame(rows)


# =========================
# PCA experiments
# =========================
pca_components_grid = [10, 15, 20, 30, 0.95]

pca_rows = []

# T on vpk
for nc in pca_components_grid:
    pca_rows.append(run_pca_cv(X2, y_T2, dataset_name="vpk", target_name="T",
                               setting_type="standard", n_components=nc,
                               class_to_min_count={0: 10}))
    pca_rows.append(run_pca_cv(X2, y_T2, dataset_name="vpk", target_name="T",
                               setting_type="ordinal", n_components=nc,
                               class_to_min_count={0: 10}))
    pca_rows.append(run_pca_cv(X2, y_T2, dataset_name="vpk", target_name="T",
                               setting_type="latent", n_components=nc,
                               class_to_min_count={0: 10}))

# N on all
for nc in pca_components_grid:
    pca_rows.append(run_pca_cv(X3, y_N3, dataset_name="all", target_name="N",
                               setting_type="standard", n_components=nc))
    pca_rows.append(run_pca_cv(X3, y_N3, dataset_name="all", target_name="N",
                               setting_type="ordinal", n_components=nc))
    pca_rows.append(run_pca_cv(X3, y_N3, dataset_name="all", target_name="N",
                               setting_type="latent", n_components=nc))

summary_pca = pd.concat(pca_rows, ignore_index=True)
summary_pca_sorted = summarize_cv_rows(summary_pca)

print("\n\n================ PCA SUMMARY ================\n")
print(summary_pca_sorted.to_string(index=False))

# =========================
# Compare PCA against non-PCA
# =========================
base = summary_phase2_sorted.copy()
pca = summary_pca_sorted.copy()

# Compare only matching dataset/target/model/setting family where possible
# We compare by dataset + target + model + broad family (standard / ordinal / latent_*)
base["family"] = base["setting"].copy()
pca["family"] = pca["setting"].str.replace(r"^pca_", "", regex=True)
pca["family"] = pca["family"].str.replace(r"_.*$", "", regex=True)  # pca_standard_10 -> standard

base_cmp = base[["dataset", "target", "model", "setting", "bal_acc_mean", "macro_f1_mean", "acc_mean"]].copy()
pca_cmp = pca[["dataset", "target", "model", "setting", "bal_acc_mean", "macro_f1_mean", "acc_mean"]].copy()

# Normalize setting names for join
base_cmp["family"] = base_cmp["setting"].str.replace(r"^latent_.*$", "latent", regex=True)
base_cmp["family"] = base_cmp["family"].str.replace(r"^(standard|ordinal)$", r"\1", regex=True)

pca_cmp["family"] = pca_cmp["setting"].str.replace(r"^pca_(standard|ordinal|latent).*?$", r"\1", regex=True)

# The pca summary has multiple PCA sizes; keep them all and compare by family separately
compare = pca_cmp.merge(
    base_cmp,
    on=["dataset", "target", "model", "family"],
    how="left",
    suffixes=("_pca", "_base"),
)

compare["delta_bal_acc"] = compare["bal_acc_mean_pca"] - compare["bal_acc_mean_base"]
compare["delta_macro_f1"] = compare["macro_f1_mean_pca"] - compare["macro_f1_mean_base"]
compare["delta_acc"] = compare["acc_mean_pca"] - compare["acc_mean_base"]

print("\n\n================ PCA VS NON-PCA DELTAS ================\n")
print(
    compare.sort_values(["dataset", "target", "family", "delta_bal_acc"], ascending=[True, True, True, False])
           [["dataset", "target", "family", "model", "setting_pca", "bal_acc_mean_pca", "bal_acc_mean_base", "delta_bal_acc", "macro_f1_mean_pca", "macro_f1_mean_base", "delta_macro_f1"]]
           .to_string(index=False)
)


==================== VPK | T | PCA=10 | STANDARD ====================

==================== VPK | T | PCA=10 | ORDINAL ====================

==================== VPK | T | PCA=10 | LATENT ====================

==================== VPK | T | PCA=15 | STANDARD ====================


/Users/raghavakarthikeyakadaveru/Desktop/Raghava/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/raghavakarthikeyakadaveru/Desktop/Raghava/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/raghavakarthikeyakadaveru/Desktop/Raghava/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/raghavakarthikeyakadaveru/Desktop/Raghava/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/raghavakarthikeyakadaveru/Desktop/Raghava/anaconda3/lib/python3.12/site-packages/sklearn/linear_model


==================== VPK | T | PCA=15 | ORDINAL ====================

==================== VPK | T | PCA=15 | LATENT ====================

==================== VPK | T | PCA=20 | STANDARD ====================

==================== VPK | T | PCA=20 | ORDINAL ====================

==================== VPK | T | PCA=20 | LATENT ====================

==================== VPK | T | PCA=30 | STANDARD ====================

==================== VPK | T | PCA=30 | ORDINAL ====================

==================== VPK | T | PCA=30 | LATENT ====================

==================== VPK | T | PCA=0.95 | STANDARD ====================

==================== VPK | T | PCA=0.95 | ORDINAL ====================

==================== VPK | T | PCA=0.95 | LATENT ====================

==================== ALL | N | PCA=10 | STANDARD ====================


/Users/raghavakarthikeyakadaveru/Desktop/Raghava/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/raghavakarthikeyakadaveru/Desktop/Raghava/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/raghavakarthikeyakadaveru/Desktop/Raghava/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/raghavakarthikeyakadaveru/Desktop/Raghava/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/raghavakarthikeyakadaveru/Desktop/Raghava/anaconda3/lib/python3.12/site-packages/sklearn/linear_model


==================== ALL | N | PCA=10 | ORDINAL ====================


...


==================== ALL | N | PCA=30 | ORDINAL ====================

==================== ALL | N | PCA=30 | LATENT ====================

==================== ALL | N | PCA=0.95 | STANDARD ====================


/Users/raghavakarthikeyakadaveru/Desktop/Raghava/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/raghavakarthikeyakadaveru/Desktop/Raghava/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/raghavakarthikeyakadaveru/Desktop/Raghava/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



==================== ALL | N | PCA=0.95 | ORDINAL ====================


/Users/raghavakarthikeyakadaveru/Desktop/Raghava/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/raghavakarthikeyakadaveru/Desktop/Raghava/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



==================== ALL | N | PCA=0.95 | LATENT ====================


================ PCA SUMMARY ================

dataset target           setting                 model  acc_mean  acc_std  bal_acc_mean  bal_acc_std  macro_f1_mean  macro_f1_std  weighted_f1_mean  weighted_f1_std  macro_precision_mean  macro_recall_mean
    all      N   pca_latent_0.95                   SVR  0.437939 0.004190      0.284969     0.002292       0.269227      0.005030          0.396329         0.002799              0.280726           0.284969
    all      N   pca_latent_0.95                 Ridge  0.367230 0.108377      0.276001     0.005526       0.240345      0.004934          0.324859         0.050609              0.261816           0.276001
    all      N   pca_latent_0.95            ElasticNet  0.367230 0.108332      0.273544     0.006823       0.236916      0.008771          0.322194         0.046629              0.255952           0.273544
    all      N   pca_latent_0.95                 ETReg  